In [1]:
import nest_asyncio
nest_asyncio.apply()  # 중첩 이벤트 루프 허용
import os
import json
import logging
import asyncio
import pandas as pd
import openai
from datetime import datetime
import sys
from typing import List, Dict, Optional
from tenacity import retry, stop_after_attempt, wait_exponential
import re

In [20]:
df = pd.read_excel('../../data/centum_data/21.11-24.6환자 CC_PI_치료계획.xlsx')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df.iloc[:,1:]

df.columns = df.columns.str.strip()
df = df[['환자번호', '날짜', 
        # 'CC', '약', '장치', '습관', '찜질', '마사지, 스트레칭', 'PI',  # 처리 됨.
        'CMO','MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading', 'Occlusion', 'OJ/OB',
        'Class', 'Midline Shift', 'Deviation', 'CR-CO', 'Tongue ridging',
        'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt',
        'Lateral excursion Protrusive excursion', 'End feel', 
    #    '치료계획', # 필요 없을 듯.
    #    'T-scan 악화/개선', 'CBCT 악화/개선', 'CBCT 판독소견' # 고유겂 NaN    
    ]]

df = df[['환자번호', '날짜',
        'CMO','MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading', 'Occlusion', 'OJ/OB',
        'Class', 'Midline Shift', 'Deviation', 'CR-CO', 'Tongue ridging',
        'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt',
        'Lateral excursion Protrusive excursion', 'End feel']]

In [21]:
df

,환자번호,날짜,CMO,MMO,Cap.pal,M.pal,Noise,Loading,Occlusion,OJ/OB,...,Midline Shift,Deviation,CR-CO,Tongue ridging,Mucosal ridging,Ultrasono,Rt,Lt,Lateral excursion Protrusive excursion,End feel
0,2301-01,2023-01-17,34mm --> mm after spray and stretch,"46mm RT M , CAP --> mm after spray and stretch",-,RT M,RT click,-,RT 안닿음,2/2,...,-,NaN,NaN,+,+,NaN,1.03 ->1.49,1.24 ->1.74,NaN,soft
1,2301-01,2023-02-01,38mm --> mm after spray and stretch,"46mm RT M , CAP --> mm after spray and stretch",-,RT) M++ Lt) M+ Temp+,RT click,-,RT 안닿음,2/2,...,-,NaN,NaN,+,+,NaN,1.03 ->1.49,1.24 ->1.74,NaN,soft
2,2301-01,2023-02-17,38mm --> mm after spray and stretch,"46mm RT M , CAP --> mm after spray and stretch",-,RT) M++ Lt) M+ Temp+,RT click,-,RT 안닿음,2/2,...,-,NaN,NaN,+,+,NaN,1.03 ->1.49,1.24 ->1.74,NaN,soft
3,2301-01,2023-03-21,40mm --> mm after spray and stretch,"48mm RT M , CAP --> mm after spray and stretch",-,RT) M+/- Lt) M+/- Temp+/-,RT click,-,RT 안닿음,2/2,...,-,NaN,NaN,+,+,NaN,1.03 ->1.49,1.24 ->1.74,NaN,soft
4,2301-01,2023-04-21,48mm --> mm after spray and stretch,48mm nopain --> 53mm after spray and stretch,-,both) tenderpoint,both) click,-,RT 안닿음,2/2,...,-,NaN,NaN,+,+,NaN,1.03 ->1.49,1.24 ->1.74,NaN,soft
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28103,2405-86,2024-05-21,20mm --> 53mm after spray and stretch,48mm Lt)M+ --> mm after spray and stretch,-,"Lt)M+(광대부근) , tenderpoint . SCM->T",넘어갈떄 Lt)Click 심함,-,567/567,1/0,...,하왼2,NaN,-,+,+,NaN,1.2 ->1.65,큰문제는 없으나 약간 두터워진 부분O,NaN,soft
28104,2405-88,2024-05-22,mm --> mm after spray and stretch,mm Lt)Cap.M+ --> mm after spray and stretch,Lt)+,"Lt)M++. SCM->T , Rt)M+ ,Both)An T+",Both)click,-,567/567,2/2,...,-,NaN,-,NaN,NaN,NaN,0.96 ->1.34,1.12 ->1.6,NaN,soft
28105,2405-89,2024-05-21,29mm --> mm after spray and stretch,43mm Lt cap+--> 55mm after spray and stretch 고착후,Lt)+,Both)M+,Lt)Popping,-,567/567,2/2,...,상왼1.5,NaN,-,+,+,NaN,0.84 ->1.21,0.85 ->1.33,NaN,soft
28106,2405-96,2024-05-29,20mm --> mm after spray and stretch,42mm Rt)+ --> mm after spray and stretch,-,NaN,both) click,-,"4567/4567 (우측 6,7 긴밀하진 않음)",3/4,...,-,오른쪽 S,NaN,+,+,NaN,0.73 -> 1.14,0.74 -> 1.25,NaN,hard
